In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import ast
import subprocess
import shutil
import zipfile
import os

HOME = Path.home()
WS = HOME / "workspace"

# refresh mounts
subprocess.run(["wb", "resource", "mount"], check=False)

# persistent output bucket
OUTROOT = WS / "agemr-data"

# find AoU genomics resource without hard-coding the annoying trailing space
GENOMICS = next(
    p / "vwb-aou-datasets-controlled-v9"
    for p in WS.glob("Data from All of Us Controlled Tier *")
    if (p / "vwb-aou-datasets-controlled-v9").exists()
)

FINAL = OUTROOT / "final_cohort_v1"
FINAL.mkdir(parents=True, exist_ok=True)

print("GENOMICS =", GENOMICS)
print("FINAL =", FINAL)

Successfully mounted workspace bucket resources.
GENOMICS = /home/dataproc/workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9
FINAL = /home/dataproc/workspace/agemr-data/final_cohort_v1


In [5]:
AGEMR = Path.home() / "workspace" / "agemr-data"

print("AGEMR =", AGEMR)
print("exists =", AGEMR.exists())

for root, dirs, files in os.walk(AGEMR):
    rel = Path(root).relative_to(AGEMR)

    # 最多看 3 层，避免输出太夸张
    if len(rel.parts) > 3:
        dirs[:] = []
        continue

    print("\nDIR:", rel)

    for f in files:
        p = Path(root) / f
        print("   ", f, p.stat().st_size)

AGEMR = /home/dataproc/workspace/agemr-data
exists = True

DIR: .
    20260930_033357_1287326871_data_bmi-000000000000.csv.gz 7645572
    20260930_033357_330349905_data_bloodPressure-000000000000.csv.gz 6859430
    20260930_033357_689390139_data_person-000000000000.csv.gz 2349402
    AllOfUsControlledTierDatasetv9_WGS_BMI_BP_pilot_20260930_033422_R.ipynb 12895

DIR: covariates
    AoU_v9_EUR_PC1_PC16.tsv 102766121
    AoU_v9_ancestry_PC1_PC16.tsv 194153415

DIR: final_cohort_v1

DIR: genotypes

DIR: genotypes/BMI_AoU_ACAF_v9
    BMI_ACAF_extraction_summary.tsv 443
    BMI_chr1.log 1444
    BMI_chr1.pgen 12611072
    BMI_chr1.psam 5892291
    BMI_chr1.pvar 250651
    BMI_chr10.log 1452
    BMI_chr10.pgen 9029517
    BMI_chr10.psam 5892291
    BMI_chr10.pvar 232642
    BMI_chr11.log 1452
    BMI_chr11.pgen 9943369
    BMI_chr11.psam 5892291
    BMI_chr11.pvar 232338
    BMI_chr12.log 1452
    BMI_chr12.pgen 8657600
    BMI_chr12.psam 5892291
    BMI_chr12.pvar 232600
    BMI_chr13.log 14

In [6]:
res = subprocess.run(
    ["wb", "resource", "list-tree"],
    capture_output=True,
    text=True
)

print(res.stdout)

├── Data from All of Us Controlled Tier 
│   ├── wgs_bmi_bp
│   ├── C2025Q4R6
│   └── vwb-aou-datasets-controlled-v9
├── JupyterLab_Spark_cluster_for_AoU_Spark_20261020
├── agemr-data
├── dataproc-staging-agemr
└── dataproc-temp-agemr



In [7]:
import pandas as pd
from pathlib import Path

DATA = Path.home() / "workspace" / "agemr-data"

person_file = next(DATA.glob("*data_person*.csv.gz"))
bmi_file = next(DATA.glob("*data_bmi*.csv.gz"))
bp_file = next(DATA.glob("*data_bloodPressure*.csv.gz"))

for name, f in [
    ("PERSON", person_file),
    ("BMI", bmi_file),
    ("BP", bp_file),
]:
    print("\n====================")
    print(name)
    print(f)
    
    tmp = pd.read_csv(f, nrows=5)
    
    print("\nColumns:")
    print(tmp.columns.tolist())
    
    display(tmp)


PERSON
/home/dataproc/workspace/agemr-data/20260930_033357_689390139_data_person-000000000000.csv.gz

Columns:
['date_of_birth', 'ethnicity', 'T_DISP_ethnicity', 'ethnicity_concept_id', 'gender', 'T_DISP_gender', 'gender_concept_id', 'person_id', 'race', 'T_DISP_race', 'race_concept_id', 'self_reported_category', 'T_DISP_self_reported_category', 'self_reported_category_concept_id', 'sex_at_birth', 'T_DISP_sex_at_birth', 'sex_at_birth_concept_id']


,date_of_birth,ethnicity,T_DISP_ethnicity,ethnicity_concept_id,gender,T_DISP_gender,gender_concept_id,person_id,race,T_DISP_race,race_concept_id,self_reported_category,T_DISP_self_reported_category,self_reported_category_concept_id,sex_at_birth,T_DISP_sex_at_birth,sex_at_birth_concept_id
0,1977-06-15 00:00:00 UTC,38003564,Not Hispanic or Latino,38003564,45878463,Woman,45878463,1123640,8527,White,8527,8527,White,8527,45878463,Female,45878463
1,1958-06-15 00:00:00 UTC,38003564,Not Hispanic or Latino,38003564,45878463,Woman,45878463,7218988,8527,White,8527,8527,White,8527,45878463,Female,45878463
2,1957-06-15 00:00:00 UTC,38003564,Not Hispanic or Latino,38003564,45878463,Woman,45878463,8259947,8527,White,8527,8527,White,8527,45878463,Female,45878463
3,1977-06-15 00:00:00 UTC,38003564,Not Hispanic or Latino,38003564,45878463,Woman,45878463,7439582,8527,White,8527,8527,White,8527,45878463,Female,45878463
4,1973-06-15 00:00:00 UTC,38003564,Not Hispanic or Latino,38003564,45878463,Woman,45878463,9014206,8527,White,8527,8527,White,8527,45878463,Female,45878463



BMI
/home/dataproc/workspace/agemr-data/20260930_033357_1287326871_data_bmi-000000000000.csv.gz

Columns:
['age_at_occurrence', 'date', 'id', 'measurement', 'T_DISP_measurement', 'measurement_type_concept_id', 'measurement_type_concept_name', 'T_DISP_measurement_type_concept_name', 'person_id', 'source_criteria_id', 'source_value', 'unit', 'T_DISP_unit', 'value_enum', 'T_DISP_value_enum', 'value_numeric', 'visit_occurrence_id', 'visit_type', 'T_DISP_visit_type']


,age_at_occurrence,date,id,measurement,T_DISP_measurement,measurement_type_concept_id,measurement_type_concept_name,T_DISP_measurement_type_concept_name,person_id,source_criteria_id,source_value,unit,T_DISP_unit,value_enum,T_DISP_value_enum,value_numeric,visit_occurrence_id,visit_type,T_DISP_visit_type
0,17,2019-03-05 15:35:06 UTC,1000732578678021,3038553,Body mass index (BMI) [Ratio],44818701,44818701,From physical examination,2179343,903124,bmi,9531,kilogram per square meter,0,No matching concept,16.4,1000000732578678,9202,Outpatient Visit
1,17,2019-04-05 17:09:58 UTC,1000284106110021,3038553,Body mass index (BMI) [Ratio],44818701,44818701,From physical examination,3109820,903124,bmi,9531,kilogram per square meter,0,No matching concept,17.8,1000000284106110,9202,Outpatient Visit
2,17,2022-04-01 17:10:03 UTC,1000742939162021,3038553,Body mass index (BMI) [Ratio],44818701,44818701,From physical examination,7216262,903124,bmi,9531,kilogram per square meter,0,No matching concept,18.2,1000000742939162,9202,Outpatient Visit
3,17,2022-06-09 17:17:01 UTC,1000446390270021,3038553,Body mass index (BMI) [Ratio],44818701,44818701,From physical examination,6438327,903124,bmi,9531,kilogram per square meter,0,No matching concept,18.4,1000000446390270,9202,Outpatient Visit
4,17,2022-04-13 13:12:42 UTC,1000936983285021,3038553,Body mass index (BMI) [Ratio],44818701,44818701,From physical examination,2475678,903124,bmi,9531,kilogram per square meter,0,No matching concept,19.6,1000000936983285,9202,Outpatient Visit



BP
/home/dataproc/workspace/agemr-data/20260930_033357_330349905_data_bloodPressure-000000000000.csv.gz

Columns:
['age_at_occurrence', 'date', 'diastolic', 'id', 'measurement_type_concept_id', 'measurement_type_concept_name', 'T_DISP_measurement_type_concept_name', 'person_id', 'status_code', 'T_DISP_status_code', 'systolic', 'visit_occurrence_id', 'visit_type', 'T_DISP_visit_type']


,age_at_occurrence,date,diastolic,id,measurement_type_concept_id,measurement_type_concept_name,T_DISP_measurement_type_concept_name,person_id,status_code,T_DISP_status_code,systolic,visit_occurrence_id,visit_type,T_DISP_visit_type
0,74,2023-03-29 15:25:08 UTC,43.5,1000448368586023,44818701,44818701,From physical examination,9305664,NaN,Unknown Range,129.5,1000000448368586,9202,Outpatient Visit
1,79,2024-04-11 20:42:29 UTC,45.5,1000184956480023,44818701,44818701,From physical examination,1440208,4.0,Normal,98.5,1000000184956480,9202,Outpatient Visit
2,71,2022-04-14 15:16:58 UTC,46.0,1000785566049019,44818701,44818701,From physical examination,5211661,3.0,Hypotensive,76.5,1000000785566049,9202,Outpatient Visit
3,64,2021-04-13 17:21:48 UTC,46.0,1000106485962023,44818701,44818701,From physical examination,6660229,4.0,Normal,92.0,1000000106485962,9202,Outpatient Visit
4,77,2024-04-10 19:52:13 UTC,46.5,1000916150386032,44818701,44818701,From physical examination,1582013,3.0,Hypotensive,88.5,1000000916150386,9202,Outpatient Visit


In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path.home() / "workspace" / "agemr-data"
FINAL = DATA / "final_cohort_v1"
FINAL.mkdir(parents=True, exist_ok=True)

person_file = next(DATA.glob("*data_person*.csv.gz"))
bmi_file = next(DATA.glob("*data_bmi*.csv.gz"))
bp_file = next(DATA.glob("*data_bloodPressure*.csv.gz"))

person = pd.read_csv(
    person_file,
    dtype={"person_id": str},
    low_memory=False
)

bmi = pd.read_csv(
    bmi_file,
    dtype={"person_id": str},
    low_memory=False
)

bp = pd.read_csv(
    bp_file,
    dtype={"person_id": str},
    low_memory=False
)

eur = pd.read_csv(
    DATA / "covariates" / "AoU_v9_EUR_PC1_PC16.tsv",
    sep="\t",
    dtype={"research_id": str},
    low_memory=False
)

print("person rows:", len(person),
      "unique people:", person.person_id.nunique())

print("BMI rows:", len(bmi),
      "unique people:", bmi.person_id.nunique())

print("BP rows:", len(bp),
      "unique people:", bp.person_id.nunique())

print("EUR:", eur.research_id.nunique())

person rows: 289902 unique people: 289902
BMI rows: 281536 unique people: 272835
BP rows: 241179 unique people: 241179
EUR: 284004


In [10]:
# Sanity check
print(person["race"].value_counts(dropna=False).head())
print(eur["ancestry_pred_other"].value_counts(dropna=False))

# person table should already be White + WGS,
# but keep the condition explicit for reproducibility
white = (
    person.loc[
        pd.to_numeric(person["race"], errors="coerce") == 8527
    ]
    .drop_duplicates("person_id")
    .copy()
)
eur = (
    eur.loc[
        eur["ancestry_pred_other"].astype(str).str.lower() == "eur"
    ]
    .drop_duplicates("research_id")
    .copy()
)

master = white.merge(
    eur,
    left_on="person_id",
    right_on="research_id",
    how="inner",
    validate="one_to_one"
)

master["IID"] = master["person_id"].astype(str)

print("White + WGS:", white.person_id.nunique())
print("Predicted EUR:", eur.research_id.nunique())
print("White ∩ EUR:", master.IID.nunique())

race
8527    289902
Name: count, dtype: int64
ancestry_pred_other
eur    284004
Name: count, dtype: int64
White + WGS: 289902
Predicted EUR: 284004
White ∩ EUR: 266676


In [11]:
pc_cols = [f"PC{i}" for i in range(1, 11)]

master_keep = master[
    [
        "IID",
        "person_id",
        "date_of_birth",
        "sex_at_birth",
        "ancestry_pred",
        "ancestry_pred_other",
    ] + pc_cols
].copy()

master_keep.to_csv(
    FINAL / "sample_master.tsv",
    sep="\t",
    index=False
)

print("Final genetic parent cohort:", master_keep["IID"].nunique())

Final genetic parent cohort: 266676


In [12]:
bmi["person_id"] = bmi["person_id"].astype(str)
bmi["date"] = pd.to_datetime(bmi["date"], errors="coerce", utc=True)
bmi["BMI"] = pd.to_numeric(bmi["value_numeric"], errors="coerce")
bmi["age"] = pd.to_numeric(bmi["age_at_occurrence"], errors="coerce")

bmi_long = (
    bmi.loc[
        bmi["BMI"].notna()
        & bmi["age"].notna()
        & (bmi["age"] >= 18)
    ]
    .merge(
        master_keep,
        left_on="person_id",
        right_on="IID",
        how="inner"
    )
    .sort_values(["IID", "date"])
)

print("BMI rows:", len(bmi_long))
print("BMI participants:", bmi_long["IID"].nunique())

bmi_long.to_csv(
    FINAL / "bmi_long.tsv.gz",
    sep="\t",
    index=False,
    compression="gzip"
)

BMI rows: 259157
BMI participants: 251110


In [13]:
bp["person_id"] = bp["person_id"].astype(str)
bp["date"] = pd.to_datetime(bp["date"], errors="coerce", utc=True)
bp["SBP"] = pd.to_numeric(bp["systolic"], errors="coerce")
bp["DBP"] = pd.to_numeric(bp["diastolic"], errors="coerce")
bp["age"] = pd.to_numeric(bp["age_at_occurrence"], errors="coerce")

bp_long = (
    bp.loc[
        bp["age"].notna()
        & (bp["age"] >= 18)
        & (bp["SBP"].notna() | bp["DBP"].notna())
    ]
    .merge(
        master_keep,
        left_on="person_id",
        right_on="IID",
        how="inner"
    )
    .sort_values(["IID", "date"])
)

print("BP rows:", len(bp_long))
print("BP participants:", bp_long["IID"].nunique())

bp_long.to_csv(
    FINAL / "bp_long.tsv.gz",
    sep="\t",
    index=False,
    compression="gzip"
)

BP rows: 221393
BP participants: 221393


In [14]:
bmi_primary = (
    bmi_long
    .drop_duplicates("IID", keep="first")
    .copy()
)

bp_primary = (
    bp_long
    .drop_duplicates("IID", keep="first")
    .copy()
)

bmi_primary.to_csv(
    FINAL / "bmi_primary_earliest.tsv",
    sep="\t",
    index=False
)

bp_primary.to_csv(
    FINAL / "bp_primary_earliest.tsv",
    sep="\t",
    index=False
)

print("BMI primary N:", len(bmi_primary))
print("BP primary N:", len(bp_primary))
print(
    "BMI-BP overlap:",
    len(set(bmi_primary.IID) & set(bp_primary.IID))
)

BMI primary N: 251110
BP primary N: 221393
BMI-BP overlap: 213607


In [15]:
print("\nBMI age distribution")
print(
    bmi_primary["age"].describe(
        percentiles=[.1, .25, .5, .75, .9]
    )
)

print("\nBP age distribution")
print(
    bp_primary["age"].describe(
        percentiles=[.1, .25, .5, .75, .9]
    )
)

from scipy.stats import ks_2samp

ks = ks_2samp(
    bmi_primary["age"].dropna(),
    bp_primary["age"].dropna()
)

print("\nKS statistic:", ks.statistic)
print("KS p-value:", ks.pvalue)


BMI age distribution
count    251110.000000
mean         55.254793
std          16.939103
min          18.000000
10%          30.000000
25%          42.000000
50%          58.000000
75%          69.000000
90%          75.000000
max         118.000000
Name: age, dtype: float64

BP age distribution
count    221393.000000
mean         55.325733
std          17.087464
min          18.000000
10%          30.000000
25%          42.000000
50%          58.000000
75%          69.000000
90%          76.000000
max         118.000000
Name: age, dtype: float64

KS statistic: 0.003786961421679391
KS p-value: 0.06828022022244384


In [16]:
keep = (
    master_keep[["IID"]]
    .drop_duplicates()
    .sort_values("IID")
)

KEEP_FILE = FINAL / "White_EUR_keep.tsv"

keep.to_csv(
    KEEP_FILE,
    sep="\t",
    index=False
)

print("Keep N =", len(keep))

Keep N = 266676


In [17]:
from pathlib import Path
import pandas as pd
import numpy as np
import subprocess
import shutil

HOME = Path.home()
DATA = HOME / "workspace" / "agemr-data"
FINAL = DATA / "final_cohort_v1"

GENO_IN = DATA / "genotypes" / "BMI_AoU_ACAF_v9"
GENO_OUT = FINAL / "genotypes_White_EUR"
GENO_OUT.mkdir(parents=True, exist_ok=True)

# find PLINK2
PLINK2 = shutil.which("plink2")

if PLINK2 is None:
    candidate = HOME / "plink2_env" / "bin" / "plink2"
    if candidate.exists():
        PLINK2 = str(candidate)

if PLINK2 is None:
    MAMBA = "/opt/conda/default/bin/mamba"
    envdir = HOME / "plink2_env"

    subprocess.run(
        [
            MAMBA, "create", "-y",
            "-p", str(envdir),
            "-c", "conda-forge",
            "-c", "bioconda",
            "plink2"
        ],
        check=True
    )

    PLINK2 = str(envdir / "bin" / "plink2")

print("PLINK2 =", PLINK2)

subprocess.run(
    [PLINK2, "--version"],
    check=True
)

Transaction

  Prefix: /home/dataproc/plink2_env

  Updating specs:

   - plink2


  Package             Version  Build                Channel          Size
───────────────────────────────────────────────────────────────────────────
  Install:
───────────────────────────────────────────────────────────────────────────

  + libzlib             1.3.2  h25fd6f3_3           conda-forge      64kB
  + libgomp            16.2.0  he0feb66_7           conda-forge     641kB
  + zstd                1.5.7  hb78ec9c_7           conda-forge     601kB
  + _openmp_mutex         4.5  20_gnu               conda-forge      29kB
  + libgcc             16.2.0  ha9f2e26_7           conda-forge       1MB
  + libgfortran5       16.2.0  h6b99dfc_7           conda-forge       3MB
  + libstdcxx          16.2.0  h934c35e_7           conda-forge       7MB
  + libgfortran        16.2.0  h69a702a_7           conda-forge      29kB
  + libopenblas        0.3.34  pthreads_hf13c14d_2  conda-forge       7MB
  + plink2   

CompletedProcess(args=['/home/dataproc/plink2_env/bin/plink2', '--version'], returncode=0)

In [18]:
master = pd.read_csv(
    FINAL / "sample_master.tsv",
    sep="\t",
    dtype={"IID": str}
)

keep = (
    master[["IID"]]
    .drop_duplicates()
    .sort_values("IID")
)

print("White ∩ EUR N =", len(keep))

KEEP_FILE = FINAL / "White_EUR_keep.tsv"

# PLINK2-friendly one-column sample file
keep.rename(
    columns={"IID": "#IID"}
).to_csv(
    KEEP_FILE,
    sep="\t",
    index=False
)

print(KEEP_FILE)

White ∩ EUR N = 266676
/home/dataproc/workspace/agemr-data/final_cohort_v1/White_EUR_keep.tsv


In [19]:
manifest = pd.read_csv(
    GENO_IN / "BMI_target_to_AoU_variant_manifest.tsv",
    sep="\t",
    dtype=str
)

print(manifest.columns.tolist())
print("Manifest rows =", len(manifest))

display(manifest.head())

['chrom', 'start0', 'end0', 'rsid', 'POS', '#CHROM', 'ID', 'REF', 'ALT']
Manifest rows = 3023


,chrom,start0,end0,rsid,POS,#CHROM,ID,REF,ALT
0,chr1,177919889,177919890,rs539515,177919890,1,chr1:177919890:A,A,"C,T"
1,chr1,201831382,201831383,rs2678204,201831383,1,chr1:201831383:T,T,G
2,chr1,72284786,72284787,rs34361149,72284787,1,chr1:72284787:G,G,"A,T"
3,chr1,109612065,109612066,rs17024393,109612066,1,chr1:109612066:T,T,C
4,chr1,49317099,49317100,rs241460,49317100,1,chr1:49317100:A,A,G


In [20]:
bases = {"A", "C", "G", "T"}

manifest["REF"] = manifest["REF"].astype("string")
manifest["ALT"] = manifest["ALT"].astype("string")

has_variant = (
    manifest["ID"].notna()
    & manifest["REF"].notna()
    & manifest["ALT"].notna()
)

is_biallelic_snv = (
    has_variant
    & manifest["REF"].isin(bases)
    & manifest["ALT"].isin(bases)
    & ~manifest["ALT"].str.contains(",", na=False)
)

strict = manifest.loc[
    is_biallelic_snv
].copy()

needs_resolution = manifest.loc[
    has_variant & ~is_biallelic_snv
].copy()

missing_acaf = manifest.loc[
    ~has_variant
].copy()

print("Total target loci:          ", len(manifest))
print("Strict biallelic SNPs:      ", len(strict))
print("Need allele resolution:     ", len(needs_resolution))
print("Missing from ACAF:          ", len(missing_acaf))

Total target loci:           3023
Strict biallelic SNPs:       1902
Need allele resolution:      1116
Missing from ACAF:           5


In [21]:
strict["effect_allele"] = strict["ALT"]
strict["other_allele"] = strict["REF"]

strict["allele_definition"] = "AoU_ALT_dosage"

strict.to_csv(
    FINAL / "variant_manifest_strict_biallelic.tsv",
    sep="\t",
    index=False
)

needs_resolution.to_csv(
    FINAL / "variant_manifest_needs_allele_resolution.tsv",
    sep="\t",
    index=False
)

missing_acaf.to_csv(
    FINAL / "variant_manifest_missing_ACAF.tsv",
    sep="\t",
    index=False
)

In [22]:
def count_psam(path):
    n = 0
    with open(path) as f:
        for line in f:
            if not line.startswith("#"):
                n += 1
    return n


def count_pvar(path):
    n = 0
    with open(path) as f:
        for line in f:
            if not line.startswith("#"):
                n += 1
    return n

In [23]:
summary = []

TMP = HOME / "agemr_final_tmp"
TMP.mkdir(exist_ok=True)

for c in range(1, 23):

    chrom = f"chr{c}"

    print(f"\n========== {chrom} ==========")

    target_chr = strict.loc[
        strict["chrom"] == chrom
    ].copy()

    ids = (
        target_chr["ID"]
        .dropna()
        .drop_duplicates()
    )

    extract_file = TMP / f"{chrom}_strict_ids.txt"

    ids.to_csv(
        extract_file,
        index=False,
        header=False
    )

    in_prefix = GENO_IN / f"BMI_{chrom}"
    out_prefix = GENO_OUT / f"BMI_{chrom}"

    cmd = [
        PLINK2,
        "--pfile", str(in_prefix),
        "--keep", str(KEEP_FILE),
        "--extract", str(extract_file),
        "--make-pgen",
        "--out", str(out_prefix)
    ]

    res = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    if res.returncode != 0:
        print(res.stdout)
        print(res.stderr)
        raise RuntimeError(f"PLINK2 failed on {chrom}")

    n_sample = count_psam(
        Path(str(out_prefix) + ".psam")
    )

    n_variant = count_pvar(
        Path(str(out_prefix) + ".pvar")
    )

    print(
        f"samples={n_sample:,}, "
        f"variants={n_variant:,}, "
        f"expected variants={len(ids):,}"
    )

    summary.append({
        "chrom": chrom,
        "samples": n_sample,
        "expected_variants": len(ids),
        "output_variants": n_variant
    })


========== chr1 ==========
samples=266,676, variants=172, expected variants=172

========== chr2 ==========
samples=266,676, variants=162, expected variants=162

========== chr3 ==========
samples=266,676, variants=143, expected variants=143

========== chr4 ==========
samples=266,676, variants=142, expected variants=142

========== chr5 ==========
samples=266,676, variants=120, expected variants=120

========== chr6 ==========
samples=266,676, variants=142, expected variants=142

========== chr7 ==========
samples=266,676, variants=94, expected variants=94

========== chr8 ==========
samples=266,676, variants=125, expected variants=125

========== chr9 ==========
samples=266,676, variants=73, expected variants=73

========== chr10 ==========
samples=266,676, variants=87, expected variants=87

========== chr11 ==========
samples=266,676, variants=84, expected variants=84

========== chr12 ==========
samples=266,676, variants=82, expected variants=82

========== chr13 ==========
sample

In [24]:
filter_summary = pd.DataFrame(summary)

display(filter_summary)

print(
    "\nUnique sample counts:",
    filter_summary["samples"].unique()
)

print(
    "Total strict target variants:",
    filter_summary["expected_variants"].sum()
)

print(
    "Total output variants:",
    filter_summary["output_variants"].sum()
)

,chrom,samples,expected_variants,output_variants
0,chr1,266676,172,172
1,chr2,266676,162,162
2,chr3,266676,143,143
3,chr4,266676,142,142
4,chr5,266676,120,120
5,chr6,266676,142,142
6,chr7,266676,94,94
7,chr8,266676,125,125
8,chr9,266676,73,73
9,chr10,266676,87,87



Unique sample counts: [266676]
Total strict target variants: 1902
Total output variants: 1902


In [25]:
assert (
    filter_summary["samples"] == 266676
).all()

assert (
    filter_summary["expected_variants"]
    ==
    filter_summary["output_variants"]
).all()

print("ALL CHECKS PASSED")

ALL CHECKS PASSED


In [26]:
filter_summary.to_csv(
    FINAL / "genotype_filter_summary.tsv",
    sep="\t",
    index=False
)

In [27]:
chr1_psam = pd.read_csv(
    GENO_OUT / "BMI_chr1.psam",
    sep=r"\s+",
    dtype=str
)

print(chr1_psam.shape)
display(chr1_psam.head())

(266676, 2)


,#IID,SEX
0,1000004,NaN
1,1000033,NaN
2,1000059,NaN
3,1000061,NaN
4,1000062,NaN


In [28]:
chr1_pvar = pd.read_csv(
    GENO_OUT / "BMI_chr1.pvar",
    sep="\t",
    comment="#",
    header=None
)

print("chr1 variants =", len(chr1_pvar))

chr1 variants = 172
